# Arcadia run 4: run 3 with twice the steps (6000), on a Kaggle T4

Same model shape as runs 1 and 2. New: book passages wrapped in `<myth>` / `<tale>` / `<saga>`
source tags (DreamBank keeps `<dream>`), 22 more books, DreamBank kept at run 2's 2,529 reports
(~14% of a 14.3M-character corpus), dropout 0.3 and early stopping after 2 evals without improvement.

Before running: **Settings → Accelerator → GPU T4**, **Internet on**, **Add-ons → Secrets → `HF_TOKEN`**
ticked. Run with **Save Version → Save & Run All**. Building the corpus downloads 43 books
(from gutenberg.org or its official mirror), so cell 4 takes a few minutes.

## 1. Clone the repo

In [ ]:
REPO = "https://github.com/DanielGregory/Arcadia"
BRANCH = "main"

import os
if not os.path.exists("/kaggle/working/Arcadia"):
    !git clone --depth 1 -b {BRANCH} {REPO} /kaggle/working/Arcadia
%cd /kaggle/working/Arcadia
!git log --oneline -1

## 2. Install only what Kaggle is missing (torch is never reinstalled)

In [ ]:
import importlib.util, subprocess, sys

missing = [pkg for pkg in ("sentencepiece", "datasets") if importlib.util.find_spec(pkg) is None]
print("installing:", missing or "nothing, all present")
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

## 3. Check the GPU

In [ ]:
import torch

print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none: set Accelerator to GPU T4")

## 4. Build the corpus with source tags, DreamBank fixed at 2,529 reports

In [ ]:
!python -m engine.data_prep.build_corpus --tags --dreambank_reports 2529

## 5. Train the tokenizer with the source tags reserved as single tokens

In [ ]:
!python -m engine.tokenizer --vocab_size 8000 --tags

## 6. Train

Settings in `configs/run4.py`. Compare the bits-per-char columns with run 3 (best 1.594 overall at step 3000).

In [ ]:
!python -m engine.train --config configs/run4.py --out_dir /kaggle/working/run4

## 7. Samples: three passages in each voice

Temperature 0.9, top-p 0.9, repetition penalty 1.1, at most 250 tokens, cut at the last full sentence.

In [ ]:
for tag in ["dream", "myth", "tale", "saga"]:
    !python -m engine.sample --ckpt /kaggle/working/run4/best.pt --dreams 3 --tag {tag} --tokens 250 --temperature 0.9

## 8. Upload to Hugging Face

Goes to `DannyGregs/arcadia-dreams/run4` (private). Skips if the `HF_TOKEN` secret isn't attached.

In [ ]:
HF_REPO = "DannyGregs/arcadia-dreams"
RUN_DIR = "/kaggle/working/run4"

import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as err:
    print("no HF_TOKEN secret, skipping upload:", err)
else:
    !python -m engine.hub push --repo {HF_REPO} --run_dir {RUN_DIR}